# Phase 4 — Speechocean762 Feature Extraction

Extracts the project's standard acoustic + linguistic feature set for every utterance in
**Speechocean762** (OpenSLR SLR101 / HF mirror `mispeech/speechocean762`), so its real,
human-rated **fluency score** (1 of 5 expert annotators, scale 1-10) can be used as a genuine
supervised regression target.

This is different from the NISQA experiment elsewhere in this project: NISQA's MOS label is
*audio quality*, not communication/fluency. Speechocean762's `fluency` column is a real fluency
rating, which is a legitimate (if narrower-than-"full communication") target. See the project
report's **Scope and limitations** section for the distinction.

Reuses the exact `pause_features` / `pitch_features` / `spectral_mfcc_features` / `linguistic_features`
/ `filler_features` functions from `metrics/compute_metrics.py` (which itself reuses
`Phase2_CommonVoice_Communication_Feature_Extraction.ipynb`'s functions), so features here are
directly comparable to the rest of the project's feature bank.

**Runtime note:** this notebook processes recordings serially with checkpointing (same pattern as
`Phase2_CommonVoice_Communication_Feature_Extraction.ipynb`), which is reliable inside Jupyter but
slow (~1.5s/clip, ~5000 clips ≈ 2 hours for the full dataset). A parallelized, multiprocessing
version of the same logic is available as `training/extract_speechocean_features.py` for a faster
run outside Jupyter (process-pool parallelism is unreliable inside Jupyter on Windows). Set
`LIMIT` below to a smaller number to test quickly before committing to a full run.

In [ ]:
import io
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa
from tqdm.auto import tqdm

sys.path.insert(0, str(Path.cwd().parent / "metrics"))
from compute_metrics import (
    pause_features, pitch_features, spectral_mfcc_features, safe_stats,
    linguistic_features, filler_features, TARGET_SR, N_FFT, HOP_LENGTH,
)

print("Imports OK")

In [ ]:
# --- Configuration ---

# Path to the downloaded Speechocean762 snapshot (huggingface_hub cache).
# Find yours with: huggingface_hub.snapshot_download(repo_id="mispeech/speechocean762", repo_type="dataset")
HF_SNAPSHOT = Path(
    r"C:\Users\Mounindra\.cache\huggingface\hub\datasets--mispeech--speechocean762"
    r"\snapshots\06385584fad212b26134c656fdd3ccf9f093f33e\data"
)

OUT_DIR = Path.cwd().parent / "data"
OUT_CSV = OUT_DIR / "speechocean762_features.csv"
CHECKPOINT_EVERY = 250

# Set to an integer (e.g. 200) to test on a small subset first; None = full dataset (~5000 utterances).
LIMIT = None

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output ->", OUT_CSV)

In [ ]:
def load_rows(limit=None):
    frames = []
    for split, fname in [("train", "train-00000-of-00001.parquet"), ("test", "test-00000-of-00001.parquet")]:
        tbl = pq.read_table(
            HF_SNAPSHOT / fname,
            columns=["fluency", "accuracy", "completeness", "prosodic", "total",
                     "text", "speaker", "gender", "age", "audio"],
        )
        df = tbl.to_pandas()
        df["split"] = split
        frames.append(df)
    full = pd.concat(frames, ignore_index=True)
    if limit:
        # Keep whole speakers intact (not a random row sample) so grouped CV later has clean groups.
        speakers = full["speaker"].unique()
        keep_speakers = set(speakers[: max(1, limit // 20)])  # ~20 utterances/speaker
        full = full[full["speaker"].isin(keep_speakers)].reset_index(drop=True)
    return full

df_raw = load_rows(LIMIT)
print(f"Loaded {len(df_raw)} utterances from {df_raw['speaker'].nunique()} unique speakers")
df_raw.head()

In [ ]:
def extract_one(row):
    try:
        y, sr = sf.read(io.BytesIO(row["audio"]["bytes"]))
        y = np.asarray(y, dtype=np.float32)
        if y.ndim > 1:
            y = y.mean(axis=1)
        if sr != TARGET_SR:
            y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)
            sr = TARGET_SR
        if len(y) == 0:
            raise ValueError("empty audio")

        total_duration = len(y) / sr
        pause = pause_features(y, sr)
        pitch = pitch_features(y, sr)
        spectral = spectral_mfcc_features(y, sr)
        text = row["text"]
        ling = linguistic_features(text)
        fill = filler_features(text)

        rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH).flatten()

        words = ling["word_count"]
        speech_duration = pause["speech_duration_seconds"]
        wpm = (words / speech_duration * 60.0) if speech_duration > 0 and words > 0 else np.nan
        wps = (words / speech_duration) if speech_duration > 0 and words > 0 else np.nan

        out = {
            "speaker": row["speaker"], "gender": row["gender"], "age": row["age"], "split": row["split"],
            "text": text, "total_duration_seconds": float(total_duration),
            "wpm_speech_duration": float(wpm) if np.isfinite(wpm) else np.nan,
            "words_per_second": float(wps) if np.isfinite(wps) else np.nan,
            "fluency": row["fluency"], "accuracy": row["accuracy"], "completeness": row["completeness"],
            "prosodic": row["prosodic"], "total": row["total"],
        }
        out.update(safe_stats(rms, "rms"))
        out.update(pause)
        out.update(pitch)
        out.update(spectral)
        out.update(ling)
        out.update(fill)
        return out
    except Exception as e:
        return {"_error": str(e)}

# Smoke test on the first utterance before running the full loop.
test_result = extract_one(df_raw.iloc[0])
print({k: v for k, v in list(test_result.items())[:8]})

In [ ]:
# --- Main extraction loop, with resumable checkpointing (same pattern as Phase 2) ---

done_rows = 0
if OUT_CSV.exists():
    existing = pd.read_csv(OUT_CSV)
    done_rows = len(existing)
    print(f"Resuming: {done_rows} utterances already extracted")

results = []
t0 = time.time()
for i in tqdm(range(done_rows, len(df_raw)), initial=done_rows, total=len(df_raw)):
    result = extract_one(df_raw.iloc[i])
    results.append(result)

    if len(results) >= CHECKPOINT_EVERY or i == len(df_raw) - 1:
        batch_df = pd.DataFrame(results)
        mode = "a" if OUT_CSV.exists() else "w"
        header = not OUT_CSV.exists()
        batch_df.to_csv(OUT_CSV, mode=mode, header=header, index=False)
        results = []

elapsed = time.time() - t0
print(f"Done in {elapsed/60:.1f} min. Wrote {OUT_CSV}")

## Result

`data/speechocean762_features.csv` now holds the full feature set (acoustic + linguistic) for
every Speechocean762 utterance, alongside its real `fluency` rating. This feeds directly into
`Phase5_Fluency_Communication_Model_Training.ipynb` for the supervised training + cross-validation
step.